In [2]:
using Random


## Model Parameter and function setup


In [ ]:
N = 3 # Countries 
J = 10000 # Number of Goods 
sigma = 2 # Elasticity 
theta = 4 # Dispersion Parameter (Comparative Advantage)
T = ones(N, 1) * 1.5 # Technology (Absolute Advantage)
tau = ones(N, N) # Trade Cost 
L = ones(N, 1) # Labor Force 
w = ones(N, 1) # Wages 

Random.seed!(1234)

TaskLocalRNG()

In [7]:
function draw_prods(N, J, theta, T) # Draws our productivities using the CDF from the EK paper 
    U = rand(J, N)
    return (-T' ./ log.(U)).^(1/theta) 
end 

z = draw_prods(N, J, theta, T) 

10000×3 Matrix{Float64}:
 1.28804   1.12576   0.85982
 1.1399    1.15088   1.62592
 2.69918   1.23896   1.26942
 0.772789  1.33199   1.53782
 1.23099   4.77452   2.01918
 1.35348   1.68815   1.30513
 1.71153   1.13335   1.34728
 2.58855   1.14833   1.12312
 1.58776   0.767054  1.13251
 1.4264    1.17466   1.78731
 ⋮                   
 0.971579  1.09128   1.23819
 1.26057   1.67914   1.94662
 1.01265   0.998359  1.46099
 0.698543  2.20369   0.924977
 1.14577   1.33779   1.29491
 1.18193   0.95708   1.14383
 1.5701    1.35274   1.31893
 1.3402    1.01939   2.47223
 1.05343   2.0511    2.32996

In [8]:
function find_trade_shares(z, tau, w) 
    costs = zeros(J, N, N)
    for n in 1:N
        for i in 1:N
            costs[:, n, i] = tau[n, i] * w[i] ./ z[:, i]
        end 
    end 
    trade_shares = zeros(N,N)
    for n in 1:N 
        for i in 1:N
            trade_shares[n, i] = mean([argmin(costs[k, n, :]) == i for k in 1:J])
        end
    end
    return trade_shares 
end

trade_shares = find_trade_shares(z, tau, w) 


3×3 Matrix{Float64}:
 0.3447  0.327  0.3283
 0.3447  0.327  0.3283
 0.3447  0.327  0.3283

In [16]:
function trade_bals(trade_shares, L, w)
    expenditure = L .* w 
    trade_matrix = trade_shares .* expenditure' # Broadcast the expenditure across the rows of trade shares 
    trade_balances = zeros(N) 
    for n in 1:N 
         exports = sum(trade_matrix[:, n])
         imports = sum(trade_matrix[n, :])
         trade_balances[n] = exports - imports
    end 
    return trade_balances
end

trade_balances = trade_bals(trade_shares, L, w)

3-element Vector{Float64}:
  0.03410000000000002
 -0.018999999999999906
 -0.015100000000000113